# Step 7 — LangGraph Orchestration + Human-in-the-Loop

Runs the full six-agent pipeline as a `StateGraph`, streaming each node's output, pausing at
the **human approval gate**, and resuming after the reviewer approves or edits.

What's new vs earlier runs:
- **27 grounded facts** (incl. price, income lines, segment/geo revenue).
- **Multimodal wiki** (transcript + financial + segment + geo + filing + news) powers retrieval.
- **Segment/YoY-driven questions** from real defeatbeta-api breakdown signals.
- **Safe Harbor** compliance section in the script; **Reg-FD lint** in verification.
- BM25-blend reranker for retrieval quality.

In production the UI is CopilotKit (`useCoAgent` / `useLangGraphInterrupt`); here we replay
the same LangGraph stream from a notebook — see `docs/ui.md`.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root(); sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print(f"ticker={settings.ticker}  period={settings.period}")
print(f"data={'mock(real cached)' if settings.use_mock_data else 'live defeatbeta-api'}  "
      f"embeddings={settings.embedding_backend}  sentiment={settings.sentiment_backend}  llm={settings.llm_backend}")

ticker=NVDA  period=FY2026Q2
data=mock(real cached)  embeddings=hash  sentiment=lexicon  llm=mock


In [2]:
from ir_copilot.graph import build_graph, initial_state
from langgraph.types import Command

graph = build_graph()
cfg   = {"configurable": {"thread_id": "nb07-demo"}}

print("=" * 65)
print("  STREAMING agent updates — pausing at the human gate")
print("=" * 65)
for ev in graph.stream(initial_state(settings.ticker, settings.period),
                       cfg, stream_mode="updates"):
    for node, upd in ev.items():
        if isinstance(upd, dict) and upd.get("messages"):
            print(f"  [{node:9}] {upd['messages'][-1][1]}")

paused = graph.get_state(cfg)
print(f"\nPAUSED before: {paused.next}  ← human-in-the-loop interrupt")
assert paused.next == ("hitl",), "expected pause at hitl"

# Inspect state at the gate
ver  = paused.values["verification"]
qs   = paused.values["predicted_questions"]
rend = paused.values["rendered"]
print(f"\n  Facts extracted  : {len(paused.values['facts'])} (incl. price, income, segment/geo)")
print(f"  Predicted Qs     : {len(qs)} (segment signals + peer lags + real analyst Qs)")
print(f"  Verification     : passed={ver.passed}  numeric={ver.numeric_violations}  compliance={ver.compliance_flags}")
print(f"  Script sections  : {[s['heading'] for s in rend['script']]}")
if qs:
    print(f"\n  Top question: {qs[0].text[:75]}")

  STREAMING agent updates — pausing at the human gate
  [extract  ] Extracted 27 grounded facts; 2 peers.


Deserializing unregistered type ir_copilot.facts.FinancialFact from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('ir_copilot.facts', 'FinancialFact')]


Deserializing unregistered type ir_copilot.agents.sentiment.SentimentSnapshot from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('ir_copilot.agents.sentiment', 'SentimentSnapshot')]


Deserializing unregistered type ir_copilot.agents.competitor.PeerComparison from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('ir_copilot.agents.competitor', 'PeerComparison')]


Deserializing unregistered type ir_copilot.agents.predictive.Question from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('ir_copilot.agents.predictive', 'Question')]


Deserializing unregistered type ir_copilot.agents.drafting.DraftBundle from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('ir_copilot.agents.drafting', 'DraftBundle')]


Deserializing unregistered type ir_copilot.agents.verify.VerifyReport from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('ir_copilot.agents.verify', 'VerifyReport')]


  [wiki     ] Indexed wiki (memory).
  [sentiment] Net sentiment 0.0; 0 concerns.
  [compare  ] Peer leads=['gross_margin', 'operating_margin', 'roe', 'roic', 'roa', 'ttm_eps'] lags=[].
  [predict  ] Predicted 8 hard questions.
  [draft    ] Drafted script/deck/Q&A (attempt 1).
  [verify   ] Verification passed.

PAUSED before: ('hitl',)  ← human-in-the-loop interrupt

  Facts extracted  : 27 (incl. price, income, segment/geo)
  Predicted Qs     : 8 (segment signals + peer lags + real analyst Qs)
  Verification     : passed=True  numeric=[]  compliance=[]
  Script sections  : ['Safe Harbor', 'Opening', 'Profitability', 'Competitive position']

  Top question: Data Center Revenue rose 92% year-over-year — what is driving that and how 


## Human reviewer: approve, edit, or regenerate

In the production CopilotKit UI this is `useLangGraphInterrupt` with an Approve / Edit /
Regenerate control.  Here we approve programmatically.

In [3]:
print("=" * 65)
print("  RESUMING with human approval")
print("=" * 65)
for ev in graph.stream(
        Command(resume=True, update={"human_feedback": {"decision": "approve"}}),
        cfg, stream_mode="updates"):
    for node, upd in ev.items():
        if isinstance(upd, dict) and upd.get("messages"):
            print(f"  [{node:9}] {upd['messages'][-1][1]}")

final = graph.get_state(cfg)
fver  = final.values["verification"]
print(f"\nFinal state: next={final.next} (empty = done)")
print(f"Verification passed: {fver.passed}")
assert final.next == () and fver.passed, "expected clean finish"

print("\nFULL PIPELINE VERIFIED:")
print("  extract(27 facts) → wiki → sentiment → compare → predict")
print("  → draft → verify(passed) → [human-gate] → finalize")

  RESUMING with human approval
  [hitl     ] Human decision: approve.
  [finalize ] Finalized artifacts: script, deck outline, Q&A cheat sheet.

Final state: next=() (empty = done)
Verification passed: True

FULL PIPELINE VERIFIED:
  extract(27 facts) → wiki → sentiment → compare → predict
  → draft → verify(passed) → [human-gate] → finalize


### Going live on the MI300X
Flip `.env`: `LLM_BACKEND=vllm`, `EMBEDDING_BACKEND=vllm`, `QDRANT_MODE=docker`,
`USE_MOCK_DATA=false`, `SENTIMENT_BACKEND=finbert`.  No agent code changes — see
`docs/rocm-vllm.md`.

**Next (Step 8):** fine-tuning the Predictive Analyst (Unsloth QLoRA) — GPU-only recipe in
`08_finetuning_unsloth.ipynb` and `docs/finetuning.md`.